In [2]:
!pip install backtesting
!pip install matplotlib

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from backtesting import Strategy
from backtesting import Backtest
import asyncio
from datetime import datetime, timedelta
import nest_asyncio
from typing import Dict, List, Optional, Any, Callable, Union
import numpy as np


import sys
from pathlib import Path

# Добавляем корень проекта в путь импортов
# Предположим, ноутбук лежит в notebooks/, а проект — на уровень выше
project_root = r"C:\Users\Khanin Maksim\PycharmProjects\honeyProject\docker\data_loader"
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Теперь импорты работают
from metrics.builtins.python_metrics import *
from T_con import TConnector
from constants import TIMEFRAMES



C:\Users\Khanin Maksim\AppData\Local\Programs\Python\Python312\Lib\site-packages\backtesting\_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

In [2]:
# ===== Ячейка 1: Импорты и настройка =====


nest_asyncio.apply()  # Критично для asyncio в Jupyter!

# Твои параметры
TOKEN = 't.YS2uyKoFJ_BjA2Jz2CLNsRrpEWL5e7ad4Mq48OKNUySiNbs2QrGhIcW4gkj4-MTl62oO1quiZK8GPLkd6OM7Dw' # 🔐 Храни в .env, не коммить!
TICKER = "SBER"
TIMEFRAME = "1d"  # Для 4 лет лучше дневки, иначе получишь 4*252*6.5*12 = ~75k свечей для 1h
YEARS = 4

# ===== Ячейка 2: Функция-обёртка для загрузки =====
async def fetch_candles_to_df(
    connector: TConnector,
    ticker: str,
    timeframe: str,
    years: int = 1
) -> pd.DataFrame:
    """
    Загружает исторические свечи и возвращает pandas DataFrame.
    """
    # 1. Получаем UID инструмента
    uid = await connector.get_instrument_uid(ticker)
    if not uid:
        raise ValueError(f"❌ Тикер {ticker} не найден")

    # 2. Рассчитываем диапазон дат
    #to_date = datetime.now()
    #from_date = to_date - timedelta(days=years * 365)

    # 3. Загружаем данные (твой метод уже умеет чанковать)
    candles = await connector.load_historical_data(
        ticker=ticker,
        uid=uid,
        timeframe=timeframe,
        history_depth_days=years * 365  # ~1460 дней
    )

    # 4. Конвертируем в DataFrame
    if not candles:
        return pd.DataFrame()

    df = pd.DataFrame(candles)
    df['time'] = pd.to_datetime(df['time'])
    df = df.set_index('time').sort_index()

    # 5. Добавляем метаданные
    df.attrs['ticker'] = ticker
    df.attrs['timeframe'] = timeframe

    return df

# ==============================================================================
# 2. ФУНКЦИЯ РАСЧЁТА (исправлена: передаёт полную историю до текущего бара)
# ==============================================================================
def calculate_metric_series(
    candles_df: pd.DataFrame,
    metric_class,
    window: int = 50,
    metric_key: Optional[str] = None,  # Один ключ → Series
    metric_keys: Optional[List[str]] = None  # Список ключей → DataFrame
) -> Union[pd.Series, pd.DataFrame]:
    """
    Прогоняет метрику по всей истории.

    Параметры:
    ----------
    candles_df : pd.DataFrame
        DataFrame с колонками ['open','high','low','close'], индекс = time
    metric_class : PythonMetric
        Класс метрики из твоего core.metrics
    window : int
        Размер окна для расчёта (переопределяет min_candles, если задан)
    metric_key : str, optional
        Если указан — извлекает только этот ключ из результата, возвращает Series
    metric_keys : List[str], optional
        Если указан список — извлекает эти ключи, возвращает DataFrame

    Возвращает:
    -----------
    pd.Series или pd.DataFrame с индексом = time
    """
    metric = metric_class()
    effective_window = metric.min_candles
    
    results = []
    candles_list = candles_df[['open','high','low','close']].to_dict('records')

    for i in range(len(candles_list)):
        # Берём скользящее окно
        start_idx = max(0, i - effective_window + 1)
        window_data = candles_list[start_idx : i + 1]
        
        if len(window_data) < metric.min_candles:
            results.append(None)
            continue
        
        res = metric.calculate_python(window_data)
        
        if not res:  # Пустой результат
            results.append(None)
            continue
        
        # Логика извлечения значений
        if metric_key is not None:
            # Один ключ → одно значение
            val = res.get(metric_key)
            results.append(val)
        elif metric_keys is not None:
            # Список ключей → dict для этой строки
            row = {key: res.get(key) for key in metric_keys}
            results.append(row)
        else:
            # Все ключи → dict для этой строки
            results.append(res)
    
    # Формируем результат
    if metric_key is not None:
        # Возвращаем Series
        return pd.Series(results, index=candles_df.index, name=metric_key or metric.name)
    else:
        # Возвращаем DataFrame
        df_result = pd.DataFrame(results, index=candles_df.index)
        # Переименовываем колонки: добавляем префикс, если нужно
        if metric_key is None and metric_keys is None:
            # Все метрики — оставляем как есть
            pass
        elif metric_keys is not None:
            # Только выбранные — оставляем как есть
            df_result = df_result[metric_keys]
        return df_result


In [3]:
# ===== Ячейка 3: Запуск =====
# Создаём коннектор
connector = TConnector(token=TOKEN, log_level="INFO")

# Загружаем данные
df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="MDMG",
    timeframe="1d",  # Дневки: ~1000 строк за 4 года — идеально для бэктеста
    years=5
)

# Смотрим результат
#print(f"✅ Загружено {len(df)} свечей")
#print(df.head())
#print(df.tail())

# Быстрая визуализация
#df['close'].plot(title=f"{TICKER} Close Price", figsize=(12, 4), grid=True)

2026-04-19 08:30:43,839 | INFO | TConnector | ✅ TConnector инициализирован. Готов сосать... то есть, работать с Tinkoff API.
2026-04-19 08:30:43,841 | INFO | TConnector | 🔄 Принудительное обновление кэша...
2026-04-19 08:30:43,842 | INFO | TConnector | 📦 Загрузка реестра инструментов из Tinkoff API...
2026-04-19 08:30:47,016 | INFO | TConnector | Загрузка индикативов...
2026-04-19 08:30:47,064 | INFO | TConnector | Индикативы: 73 инструментов
2026-04-19 08:30:47,065 | INFO | TConnector | ✅ Всего загружено: 4186 инструментов
2026-04-19 08:30:47,068 | INFO | TConnector | ✅ Кэш обновлен: 4182 инструментов
2026-04-19 08:30:47,069 | INFO | TConnector | ✅ Найден UID для MDMG: 0d53d29a-3794-41c6-ba72-556d46bacb46
2026-04-19 08:30:47,070 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-19 08:30:47,145 | INFO | TConnector | 📥 Кусок [04-20 08:30 - 04-20 08:30]: 235 свечей. Всего: 235
2026-04-19 08:30:47,493 | INFO | TConnector | 📥 Кусок [04-20 08:30 - 04-20 08:30]: 25

In [6]:
{'skew_200': 0.3799, 'kurt_excess_200': -0.2275}

{'skew_200': 0.3799, 'kurt_excess_200': -0.2275}

In [7]:
from metrics.builtins.python_metrics import ZScoreMetric
from metrics.builtins.python_metrics import SkewKurtosisMetric


df = df_raw.copy()

def calculate_skewness(series, window=20):
    """Скользящая скошенность (Skewness)"""

    def skew_func(x):
        if len(x) < 3:
            return 0
        return pd.Series(x).skew()

    return series.rolling(window=window, min_periods=1).apply(skew_func, raw=False)

def calculate_kurtosis(series, window=20):
    """Скользящий эксцесс (Kurtosis) — Fisher (норма = 0)"""

    def kurt_func(x):
        if len(x) < 4:
            return 0
        return pd.Series(x).kurtosis()

    return series.rolling(window=window, min_periods=1).apply(kurt_func, raw=False)

# 2. Считаешь метрики разово (pandas тут разрешён, ядро не трогаем)
df['skew_200'] = calculate_metric_series(df_raw, SkewKurtosisMetric, metric_key='skew_200')
df['kurt_200'] = calculate_metric_series(df, SkewKurtosisMetric, metric_key='kurt_excess_200')  # или отдельный класс
df['zscore_200'] = calculate_metric_series(df, ZScoreMetric, metric_key='z_score_200')

df['skew_200'] = calculate_skewness(df['close'], 100)
df['kurt_200'] = calculate_kurtosis(df['close'], 100)

df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})

# 3. Отбрасываем NaN от warmup
df = df.dropna(subset=['skew_200', 'kurt_200', 'zscore_200'])

In [8]:
df = df_raw.copy()

def calculate_skewness(series, window=20):
    """Скользящая скошенность (Skewness)"""

    def skew_func(x):
        if len(x) < 3:
            return 0
        return pd.Series(x).skew()

    return series.rolling(window=window, min_periods=1).apply(skew_func, raw=False)

def calculate_kurtosis(series, window=20):
    """Скользящий эксцесс (Kurtosis) — Fisher (норма = 0)"""

    def kurt_func(x):
        if len(x) < 4:
            return 0
        return pd.Series(x).kurtosis()

    return series.rolling(window=window, min_periods=1).apply(kurt_func, raw=False)

def calculate_rolling_z_score(series, window=20):
    """Скользящий Z-score"""
    rolling_mean = calculate_rolling_mean(series, window)
    rolling_std = calculate_rolling_std(series, window)
    rolling_std = rolling_std.replace(0, np.nan)
    z_score = (series - rolling_mean) / rolling_std
    return z_score

def calculate_rolling_mean(series, window=20):
    """Скользящая средняя"""
    return series.rolling(window=window, min_periods=1).mean()


def calculate_rolling_std(series, window=20):
    """Скользящее стандартное отклонение (σ)"""
    return series.rolling(window=window, min_periods=1).std()


def calculate_rolling_mad(series, window=20):
    """
    Скользящее среднее абсолютное отклонение (MAD) — по Талебу
    """

    def mad_func(x):
        if len(x) < 2:
            return 0
        mean = x.mean()
        return np.mean(np.abs(x - mean))

    return series.rolling(window=window, min_periods=1).apply(mad_func, raw=False)



df['skew'] = calculate_skewness(df['close'], 100)
df['kurt'] = calculate_kurtosis(df['close'], 100)
df['zscore'] = calculate_rolling_z_score(df['close'], 100)

df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})



df = df.dropna(subset=['skew', 'kurt', 'zscore'])

In [9]:
df = df_raw.copy()

def calculate_skewness(series, window=20):
    """Скользящая скошенность (Skewness)"""

    def skew_func(x):
        if len(x) < 3:
            return 0
        return pd.Series(x).skew()

    return series.rolling(window=window, min_periods=1).apply(skew_func, raw=False)

def calculate_kurtosis(series, window=20):
    """Скользящий эксцесс (Kurtosis) — Fisher (норма = 0)"""

    def kurt_func(x):
        if len(x) < 4:
            return 0
        return pd.Series(x).kurtosis()

    return series.rolling(window=window, min_periods=1).apply(kurt_func, raw=False)

def calculate_rolling_z_score(series, window=20):
    """Скользящий Z-score"""
    rolling_mean = calculate_rolling_mean(series, window)
    rolling_std = calculate_rolling_std(series, window)
    rolling_std = rolling_std.replace(0, np.nan)
    z_score = (series - rolling_mean) / rolling_std
    return z_score

def calculate_rolling_mean(series, window=20):
    """Скользящая средняя"""
    return series.rolling(window=window, min_periods=1).mean()


def calculate_rolling_std(series, window=20):
    """Скользящее стандартное отклонение (σ)"""
    return series.rolling(window=window, min_periods=1).std()


def calculate_rolling_mad(series, window=20):
    """
    Скользящее среднее абсолютное отклонение (MAD) — по Талебу
    """

    def mad_func(x):
        if len(x) < 2:
            return 0
        mean = x.mean()
        return np.mean(np.abs(x - mean))

    return series.rolling(window=window, min_periods=1).apply(mad_func, raw=False)



df['skew'] = calculate_skewness(df['close'], 100)
df['kurt'] = calculate_kurtosis(df['close'], 100)
df['zscore'] = calculate_rolling_z_score(df['close'], 100)

df = df.rename(columns={
    'open': 'Open',
    'high': 'High',
    'low': 'Low',
    'close': 'Close',
    'volume': 'Volume'
})



df = df.dropna(subset=['skew', 'kurt', 'zscore'])

def ret_series(series):
  return pd.Series(series)

class StatArbStrategy(Strategy):

    # Параметры оптимизации
    skew_threshold = 0.1
    kurt_threshold = 1.0
    zscore_entry = 1.0
    

    def init(self,  *args, **kwargs):  

        self.lots = 0.3
        self.direction = None


        self.i_skew = self.I(ret_series,  self.data.skew)
        self.i_z_score = self.I(ret_series,  self.data.zscore)
        self.i_kurt = self.I(ret_series,  self.data.kurt)

        
    def _check_open_buy(self):

        if self.i_z_score[-1] > self.zscore_entry  and self.i_skew[-1] > self.skew_threshold:
          return True
        return False

    def _check_open_sell(self):
        if self.i_z_score[-1] < -self.zscore_entry  and self.i_skew[-1] < -self.skew_threshold:
          return True
        return False

    def _check_close_sell(self):
      if self.i_skew[-1] > self.skew_threshold or self.i_z_score[-1] > self.zscore_entry:
         return True
      return False

    def _check_close_buy(self):
      if  self.i_skew[-1] < -self.skew_threshold or self.i_z_score[-1] < -self.zscore_entry:
          return True
      return False

    def next(self):

        if self.position:

          
          if self.position.is_long and self._check_close_buy():
            self.position.close()
          if self.position.is_short and self._check_close_sell():
            self.position.close()   

        if not self.position: 
            if self._check_open_buy():
              #print(f"Сделал  лонг. Последняя цена = {self.data.df.Close[-1]}, {self.data.df.index[-1]}")
              if self.direction != "SELL_ONLY":
                self.buy(size=self.lots)

            if self._check_open_sell():
              #print(f"Сделал  шорт. Последняя цена = {self.data.df.Close[-1]}, {self.data.df.index[-1]}")
              if self.direction != "BUY_ONLY":
                self.sell(size=self.lots) 

# 4. Запускаешь бэктест
bt = Backtest(df, StatArbStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

GridPlot(id='p1446', ...)

In [10]:
df = df_raw.copy()

# ===== 1. ДОХОДНОСТИ & БАЗОВЫЕ ФУНКЦИИ =====
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

def calculate_skewness(series, window=100):
    return series.rolling(window=window, min_periods=window).apply(lambda x: pd.Series(x).skew(), raw=False)

def calculate_kurtosis(series, window=100):
    return series.rolling(window=window, min_periods=window).apply(lambda x: pd.Series(x).kurtosis(), raw=False)

def calculate_rolling_z_score(series, window=100):
    rm = series.rolling(window=window, min_periods=window).mean()
    rs = series.rolling(window=window, min_periods=window).std().replace(0, np.nan)
    return (series - rm) / rs

def calculate_roc(series, window=20):
    prev = series.shift(window)
    return (series - prev) / prev

def calculate_atr(df, window=14):
    """Исправленный ATR с защитой от NaN"""
    high, low, prev_close = df['high'], df['low'], df['close'].shift(1)
    tr = pd.concat([high - low, abs(high - prev_close), abs(low - prev_close)], axis=1).max(axis=1)
    return tr.rolling(window=window, min_periods=window).mean()

def calculate_trend_strength(df, window=50):
    def slope(arr):
        if len(arr) < 10: return np.nan
        x_local = np.arange(len(arr))
        mean_x, mean_y = x_local.mean(), arr.mean()
        num = ((x_local - mean_x) * (arr - mean_y)).sum()
        den = ((x_local - mean_x) ** 2).sum()
        return num / den if den != 0 else 0
    slopes = np.log(df['close']).rolling(window=window, min_periods=window).apply(slope, raw=False)
    vol = df['log_ret'].rolling(window=window, min_periods=window).std()
    return slopes / vol.replace(0, np.nan)

# ===== 2. РАСЧЁТ МЕТРИК (ИСПРАВЛЕНО: ATR добавлен) =====
df['skew']          = calculate_skewness(df['log_ret'], 100)
df['kurt']          = calculate_kurtosis(df['log_ret'], 100)
df['zscore']        = calculate_rolling_z_score(df['close'], 100)
df['ROC']           = calculate_roc(df['close'], 20)
df['ATR']           = calculate_atr(df, 14)           # <-- ВОТ ЭТОГО НЕ ХВАТАЛО
df['trend_strength']= calculate_trend_strength(df, 50)
df['EMA20']         = df['close'].ewm(span=20, adjust=False).mean()
df['EMA50']         = df['close'].ewm(span=50, adjust=False).mean()

# ===== 3. РЕЖИМЫ И ФИЛЬТРЫ =====
df['regime'] = np.where((df['close'] > df['EMA50']) & (df['EMA20'] > df['EMA50']), 1, -1)
df['strong_trend'] = df['trend_strength'].abs() > 0.4
df['safe'] = (df['kurt'].abs() < 5) & (df['ROC'].abs() < 0.1) & ~df['strong_trend']

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
df = df.dropna(subset=['skew','kurt','zscore','regime','ROC','trend_strength','ATR','safe'])

# ===== 4. СТРАТЕГИЯ =====
def ret_series(s): return pd.Series(s)

class StatArbStrategy(Strategy):
    zscore_entry   = 1.2
    roc_confirm    = 0.01
    atr_mult_sl    = 2.5
    atr_mult_tp_mr = 2.0
    atr_mult_tp_tr = 4.0
    vol_target     = 0.015
    max_size       = 0.4

    def init(self, *args, **kwargs):
        self.direction = None
        self.i_skew      = self.I(ret_series, self.data.skew)
        self.i_kurt      = self.I(ret_series, self.data.kurt)
        self.i_z         = self.I(ret_series, self.data.zscore)
        self.i_reg       = self.I(ret_series, self.data.regime)
        self.i_roc       = self.I(ret_series, self.data.ROC)
        self.i_safe      = self.I(ret_series, self.data.safe)
        self.i_trend_str = self.I(ret_series, self.data.trend_strength)
        self.i_atr       = self.I(ret_series, self.data.ATR)  # Теперь работает
        self.peak = self.trough = None

    def _calc_size(self):
        v = self.i_atr[-1] / self.data.Close[-1] if self.i_atr[-1] else 0.02
        return min(self.vol_target / max(v, 1e-6), self.max_size)

    def _update_trailing(self):
        if not self.position: return
        if self.position.is_long and (self.peak is None or self.data.Close[-1] > self.peak):
            self.peak = self.data.Close[-1]
        if self.position.is_short and (self.trough is None or self.data.Close[-1] < self.trough):
            self.trough = self.data.Close[-1]

    def _check_exit(self):
        if not self.position: return False
        price = self.data.Close[-1]
        atr = self.i_atr[-1] if self.i_atr[-1] else price * 0.02
        if not self.i_safe[-1]: return True
        
        entry = price - (self.position.pl / self.position.size) if self.position.size != 0 else price
        mult_tp = self.atr_mult_tp_tr if self.i_reg[-1] == 1 else self.atr_mult_tp_mr
        
        if self.position.is_long:
            trail = (self.peak or entry) - atr * mult_tp
            hard  = entry - atr * self.atr_mult_sl
            return price < max(trail, hard)
        if self.position.is_short:
            trail = (self.trough or entry) + atr * mult_tp
            hard  = entry + atr * self.atr_mult_sl
            return price > min(trail, hard)
        return False

    def next(self):
        if not self.position:
            self.peak = self.trough = None
        self._update_trailing()
        
        if self.position and self._check_exit():
            self.position.close()
            return
        
        if not self.position and self.i_safe[-1]:
            size = self._calc_size()
            reg = self.i_reg[-1]
            z = self.i_z[-1]
            roc = self.i_roc[-1]
            
            if reg == -1:  # MEAN REVERSION
                if self.i_trend_str[-1] and abs(self.i_trend_str[-1]) > 0.4: return
                if self.direction != "SELL_ONLY" and z < -self.zscore_entry and roc > self.roc_confirm:
                    self.buy(size=size)
                elif self.direction != "BUY_ONLY" and z > self.zscore_entry and roc < -self.roc_confirm:
                    self.sell(size=size)
            elif reg == 1:  # TREND
                if self.direction != "SELL_ONLY" and self.data.Close[-1] > self.data.EMA50[-1] and self.data.Close[-1] > self.data.Close[-2]:
                    self.buy(size=size)
                elif self.direction != "BUY_ONLY" and self.data.Close[-1] < self.data.EMA50[-1] and self.data.Close[-1] < self.data.Close[-2]:
                    self.sell(size=size)

# ===== 5. ЗАПУСК =====
bt = Backtest(df, StatArbStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

GridPlot(id='p2150', ...)

In [11]:
df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="MDMG",
    timeframe="1d",  # Дневки: ~1000 строк за 4 года — идеально для бэктеста
    years=5
)

2026-04-18 08:29:59,973 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-18 08:30:00,052 | INFO | TConnector | 📥 Кусок [04-19 08:29 - 04-19 08:29]: 235 свечей. Всего: 235
2026-04-18 08:30:00,340 | INFO | TConnector | 📥 Кусок [04-19 08:29 - 04-19 08:29]: 253 свечей. Всего: 488
2026-04-18 08:30:00,651 | INFO | TConnector | 📥 Кусок [04-19 08:29 - 04-18 08:29]: 255 свечей. Всего: 743
2026-04-18 08:30:00,982 | INFO | TConnector | 📥 Кусок [04-18 08:29 - 04-18 08:29]: 247 свечей. Всего: 990
2026-04-18 08:30:01,314 | INFO | TConnector | 📥 Кусок [04-18 08:29 - 04-18 08:29]: 326 свечей. Всего: 1316
2026-04-18 08:30:01,516 | INFO | TConnector | ✅ Загрузка для MDMG завершена. Всего: 1316 свечей


In [1]:
df = df_raw.copy()

# ===== 1. МЕТРИКИ (СЧИТАЕМ В PANDAS) =====
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

def calculate_skewness(series, window=100):
    return series.rolling(window=window, min_periods=window).apply(lambda x: pd.Series(x).skew(), raw=False)

def calculate_kurtosis(series, window=100):
    return series.rolling(window=window, min_periods=window).apply(lambda x: pd.Series(x).kurtosis(), raw=False)

def calculate_rolling_z_score(series, window=100):
    rm = series.rolling(window=window, min_periods=window).mean()
    rs = series.rolling(window=window, min_periods=window).std().replace(0, np.nan)
    return (series - rm) / rs

def calculate_roc(series, window=20):
    prev = series.shift(window)
    return (series - prev) / prev

def calculate_atr(df, window=14):
    high, low, prev_close = df['high'], df['low'], df['close'].shift(1)
    tr = pd.concat([high - low, abs(high - prev_close), abs(low - prev_close)], axis=1).max(axis=1)
    return tr.rolling(window=window, min_periods=window).mean()

def calculate_pullback(series, window=20):
    rolling_max = series.rolling(window=window, min_periods=1).max()
    return (rolling_max - series) / rolling_max

# ===== 2. РАСЧЁТ СКОЛЬЗЯЩИХ =====
df['skew']      = calculate_skewness(df['log_ret'], 100)
df['kurt']      = calculate_kurtosis(df['log_ret'], 100)
df['zscore']    = calculate_rolling_z_score(df['close'], 100)
df['ROC']       = calculate_roc(df['close'], 20)
df['ATR']       = calculate_atr(df, 14)
df['pullback']  = calculate_pullback(df['close'], 20)

# EMA50 для входа
df['EMA50']     = df['close'].ewm(span=50, adjust=False).mean()
# EMA100 для выхода (чтобы не выбивало на шуме)
df['EMA100']    = df['close'].ewm(span=100, adjust=False).mean()

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
df = df.dropna(subset=['EMA50','EMA100','pullback','kurt','zscore','ROC','ATR'])

# ===== 3. СТРАТЕГИЯ =====
def ret_series(s): return pd.Series(s)

class DumbTrendStrategy(Strategy):
    pullback_threshold = 0.015
    min_hold_bars      = 10
    atr_mult_sl        = 2.5      # <-- НОВОЕ: стоп-лосс = вход - 2.5*ATR
    atr_mult_tp        = 6.0      # <-- НОВОЕ: трейлинг = пик - 6.0*ATR
    
    def init(self, *args, **kwargs):
        self.i_ema50    = self.I(ret_series, self.data.EMA50)
        self.i_ema_exit = self.I(ret_series, self.data.EMA100)
        self.i_pullback = self.I(ret_series, self.data.pullback)
        self.i_atr      = self.I(ret_series, self.data.ATR)  # <-- Добавляем ATR
        self.entry_price = None
        self.peak_price  = None
        self.entry_bar   = None
        self.lots =0.3

    def next(self):
        # === ВХОД ===
        if not self.position:
            # 1. Фильтр волатильности: не входим, если дневная вола > 4%
            current_vol = self.i_atr[-1] / self.data.Close[-1] if self.i_atr[-1] else 0
            if current_vol > 0.04:
                return  # Пропускаем вход: слишком шумно
            
            # 2. Фильтр "силы тренда": цена должна быть НАД обеими скользящими минимум 10 баров
            ema50_ok = all(self.data.Close[-i] > self.i_ema50[-i] for i in range(1, min(11, len(self.data))))
            ema100_ok = all(self.data.Close[-i] > self.i_ema_exit[-i] for i in range(1, min(11, len(self.data))))
            if not (ema50_ok and ema100_ok):
                return  # Пропускаем: тренд не подтверждён
            
            # 3. Только потом — твоя логика входа
            if (self.data.Close[-1] > self.i_ema50[-1] and 
                self.data.Close[-1] > self.i_ema_exit[-1] and 
                self.i_pullback[-1] >= self.pullback_threshold):
                self.buy(size=0.3)
                self.entry_price = self.data.Close[-1]
                self.peak_price  = self.data.Close[-1]
                self.entry_bar   = len(self.data)
        
        # === ВЫХОД ===
        elif self.entry_bar is not None:
            current_atr = self.i_atr[-1] if self.i_atr[-1] else self.data.Close[-1] * 0.03
            
            # Обновляем пик для трейлинга
            if self.data.Close[-1] > self.peak_price:
                self.peak_price = self.data.Close[-1]
            
            # 1. Аварийный стоп-лосс (жесткий, от цены входа)
            hard_sl = self.entry_price - current_atr * self.atr_mult_sl
            
            # 2. Трейлинг-стоп (от пика)
            trail_stop = self.peak_price - current_atr * self.atr_mult_tp
            
            # 3. Выход по смене тренда (только после min_hold_bars)
            trend_exit = (len(self.data) - self.entry_bar) >= self.min_hold_bars and \
                         self.data.Close[-1] < self.i_ema_exit[-1]
            
            # Выходим, если сработал ЛЮБОЙ из условий
            if self.data.Close[-1] < max(hard_sl, trail_stop) or trend_exit:
                self.position.close()
                self.entry_price = self.peak_price = self.entry_bar = None
                self.entry_price = self.data.Close[-1]
                self.peak_price  = self.data.Close[-1]
                self.entry_bar   = len(self.data)

# ===== 4. ЗАПУСК =====
bt = Backtest(df, DumbTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

NameError: name 'df_raw' is not defined

In [37]:
df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="SBER",
    timeframe="1d",  # Дневки: ~1000 строк за 4 года — идеально для бэктеста
    years=5
)

2026-04-14 11:54:07,012 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-14 11:54:07,096 | INFO | TConnector | 📥 Кусок [04-15 11:54 - 04-15 11:54]: 239 свечей. Всего: 239
2026-04-14 11:54:07,388 | INFO | TConnector | 📥 Кусок [04-15 11:54 - 04-15 11:54]: 252 свечей. Всего: 491
2026-04-14 11:54:07,726 | INFO | TConnector | 📥 Кусок [04-15 11:54 - 04-14 11:54]: 253 свечей. Всего: 744
2026-04-14 11:54:08,061 | INFO | TConnector | 📥 Кусок [04-14 11:54 - 04-14 11:54]: 266 свечей. Всего: 1010
2026-04-14 11:54:08,423 | INFO | TConnector | 📥 Кусок [04-14 11:54 - 04-14 11:54]: 329 свечей. Всего: 1339
2026-04-14 11:54:08,625 | INFO | TConnector | ✅ Загрузка для SBER завершена. Всего: 1339 свечей


In [12]:
df = df_raw.copy()

# ===== 1. МИНИМУМ МЕТРИК =====
def calculate_pullback(series, window=20):
    rolling_max = series.rolling(window=window, min_periods=1).max()
    return (rolling_max - series) / rolling_max

# ===== 2. РАСЧЁТ =====
df['pullback'] = calculate_pullback(df['close'], 20)
df['EMA50']    = df['close'].ewm(span=50, adjust=False).mean()
df['SMA50']    = df['close'].rolling(window=50, min_periods=1).mean()
df['std']    = df['close'].rolling(window=50, min_periods=1).std()
df['skew'] = calculate_skewness(df['close'], 50)

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
#df = df.loc[df.index>"2022-01-10"]
df = df.dropna(subset=['EMA50','pullback'])

# ===== 3. СТРАТЕГИЯ: "DUMB & ROBUST" + TSL =====
def ret_series(s): return pd.Series(s)

class TrailingTrendStrategy(Strategy):
    # ПАРАМЕТРЫ
    min_pullback  = 0.03
    hard_stop     = 0.08
    cooldown_bars = 10
    TSL           = 0.05  # Trailing Stop Loss: 5% от текущей цены
    
    def init(self, *args, **kwargs):
        self.i_ema50    = self.I(ret_series, self.data.EMA50)
        self.i_sma50    = self.I(ret_series, self.data.SMA50)
        self.i_std = self.I(ret_series, self.data.std)
        #self.i_skew = self.I(ret_series, self.data.skew)
        self.i_pullback = self.I(ret_series, self.data.pullback)
        self.entry_price = None
        self.current_sl  = None  # Текущий уровень стопа (динамический)
        self.last_exit_bar = -100
        self.lots = 0.99
        self.std_filter = 5

    def next(self):
        # === ВХОД ===
        if not self.position:
            # 1. Кулдаун
            if len(self.data) - self.last_exit_bar < self.cooldown_bars:
                return
            # 2. Откат
            if self.i_pullback[-1] < self.min_pullback: return
            # 3. Цена выше EMA50
            if self.data.Close[-1] <= self.i_ema50[-1]: return
            # 4. Зелёная свеча
            if self.data.Close[-1] <= self.data.Open[-1]: return
            #if self.i_std[-1] < self.std_filter: return
                
            
            self.buy(size=self.lots)
            self.entry_price = self.data.Close[-1]
            self.current_sl  = self.entry_price * (1 - self.hard_stop)  # Инициализируем жёсткий стоп
            
        # === ВЫХОД / УПРАВЛЕНИЕ ПОЗИЦИЕЙ ===
        elif self.entry_price is not None:
            price = self.data.Close[-1]
            pnl_pct = (price - self.entry_price) / self.entry_price
            
            # 🔥 ЛОГИКА TRAILING STOP
            # Как только профит >= TSL (5%), активируем трейл
            if pnl_pct >= self.TSL:
                # Рассчитываем новый стоп: текущая цена - TSL%
                new_sl = price * (1 - self.TSL)
                # Двигаем стоп ТОЛЬКО вверх
                if new_sl > self.current_sl:
                    self.current_sl = new_sl
            
            # 1. Выход по трейлинг-стопу (или начальному hard stop)
            if price < self.current_sl:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return
                
            # 2. Выход по пробою EMA50 вниз (резервный фильтр)
            if price < self.i_ema50[-1]:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return

# ===== 4. ЗАПУСК =====
bt = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

trades = stats['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats.get('Max. Drawdown [%]', 0):.1f}%")
    
    # Покажи, сколько сделок закрылось по каскадному трейлу
    # (в backtesting.py нет прямой метки, но можно оценить по длительности и профиту)
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.12)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-12%): {len(spike_trades)}")

📊 Сделок: 24
💰 Win rate: 37.5%
🔁 Profit Factor: 2.41
📈 Total Return: 83.9%
📉 Max Drawdown: -26.8%
🎯 Сделок в диапазоне 'скачка' (3-12%): 1


In [39]:
typical

time
2021-04-15    1751.0800
2021-04-16    1771.0025
2021-04-19    1776.3325
2021-04-20    1772.8750
2021-04-21    1786.5075
                ...    
2026-04-10    4760.9750
2026-04-11    4749.9700
2026-04-12    4751.7475
2026-04-13    4721.9375
2026-04-14    4757.3625
Length: 1412, dtype: float64

In [13]:
df = df_raw.copy()

# ===== 1. МИНИМУМ МЕТРИК =====
def calculate_pullback(series, window=20):
    rolling_max = series.rolling(window=window, min_periods=1).max()
    return (rolling_max - series) / rolling_max

# ===== 2. РАСЧЁТ =====
df['pullback'] = calculate_pullback(df['close'], 20)
df['EMA50']    = df['close'].ewm(span=50, adjust=False).mean()
typical = (df['close']+df['open']+df['high']+df['low'])/4
df['SMA50']    = typical.rolling(window=50, min_periods=1).mean()
df['std']    = df['close'].rolling(window=50, min_periods=1).std()/df['close'][-1]
df['skew'] = calculate_skewness(df['close'], 50)

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
#df = df.loc[df.index>"2022-01-10"]
df = df.dropna(subset=['EMA50','pullback'])

# ===== 3. СТРАТЕГИЯ: "DUMB & ROBUST" + TSL =====
def ret_series(s): return pd.Series(s)

class TrailingTrendStrategy(Strategy):
    # ПАРАМЕТРЫ
    min_pullback  = 0.01
    hard_stop     = 0.08
    cooldown_bars = 1
    TSL           = 0.05  # Trailing Stop Loss: 5% от текущей цены
    
    def init(self, *args, **kwargs):
        self.i_ema50    = self.I(ret_series, self.data.EMA50)
        self.i_sma50    = self.I(ret_series, self.data.SMA50)
        self.i_std = self.I(ret_series, self.data.std)
        #self.i_skew = self.I(ret_series, self.data.skew)
        self.i_pullback = self.I(ret_series, self.data.pullback)
        self.entry_price = None
        self.current_sl  = None  # Текущий уровень стопа (динамический)
        self.last_exit_bar = -100
        self.lots = 0.99
        self.std_filter = 0.02

    def next(self):
        # === ВХОД ===
        if not self.position:
            # 1. Кулдаун
            if len(self.data) - self.last_exit_bar < self.cooldown_bars:
                return
            # 2. Откат
            if self.i_pullback[-1] < self.min_pullback: return
            # 3. Цена выше EMA50
            if self.data.Close[-1] <= self.i_sma50[-1]: return
            # 4. Зелёная свеча
            if self.data.Close[-1] <= self.data.Open[-1]: return
            if self.i_std[-1] < self.std_filter: return
                
            
            self.buy(size=self.lots)
            self.entry_price = self.data.Close[-1]
            self.current_sl  = self.entry_price * (1 - self.hard_stop)  # Инициализируем жёсткий стоп
            
        # === ВЫХОД / УПРАВЛЕНИЕ ПОЗИЦИЕЙ ===
        elif self.entry_price is not None:
            price = self.data.Close[-1]
            pnl_pct = (price - self.entry_price) / self.entry_price
            
            # 🔥 ЛОГИКА TRAILING STOP
            # Как только профит >= TSL (5%), активируем трейл
            if pnl_pct >= self.TSL:
                # Рассчитываем новый стоп: текущая цена - TSL%
                new_sl = price * (1 - self.TSL)
                # Двигаем стоп ТОЛЬКО вверх
                if new_sl > self.current_sl:
                    self.current_sl = new_sl
            
            # 1. Выход по трейлинг-стопу (или начальному hard stop)
            if price < self.current_sl:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return
                
            # 2. Выход по пробою EMA50 вниз (резервный фильтр)
            if price < self.i_sma50[-1]:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return

# ===== 4. ЗАПУСК =====
bt = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

trades = stats['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats.get('Max. Drawdown [%]', 0):.1f}%")
    
    # Покажи, сколько сделок закрылось по каскадному трейлу
    # (в backtesting.py нет прямой метки, но можно оценить по длительности и профиту)
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.12)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-12%): {len(spike_trades)}")

C:\Users\Khanin Maksim\AppData\Local\Temp\ipykernel_6476\2713189067.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df['std']    = df['close'].rolling(window=50, min_periods=1).std()/df['close'][-1]


📊 Сделок: 27
💰 Win rate: 55.6%
🔁 Profit Factor: 2.40
📈 Total Return: 90.6%
📉 Max Drawdown: -37.4%
🎯 Сделок в диапазоне 'скачка' (3-12%): 5


In [69]:
from metrics.builtins.python_metrics import Pullback20Metric, EMA50Metric

df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="SBER",
    timeframe="1d",  # Дневки: ~1000 строк за 4 года — идеально для бэктеста
    years=5
)

df = df_raw.copy()

# Рассчитываем метрики через обёртку
df['pullback_20'] = calculate_metric_series(df, Pullback20Metric, window=20, metric_key='pullback_20')
df['ema_50']      = calculate_metric_series(df, EMA50Metric, window=100, metric_key='ema_50')


#df['pullback_20'] = calculate_pullback(df['close'], 20)
#df['ema_50']    = df['close'].ewm(span=50, adjust=False).mean()

# Переименовываем под backtesting и отрезаем NaN
df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
#df = df.loc[df.index>"2022-01-10"]
df = df.dropna(subset=['ema_50', 'pullback_20'])


# ==============================================================================
# 4. СТРАТЕГИЯ (параметры синхронизированы с эталоном)
# ==============================================================================
# ==============================================================================
# 4. СТРАТЕГИЯ (параметры синхронизированы с эталоном)
# ==============================================================================
def ret_series(s): return pd.Series(s)

class TrailingTrendStrategy(Strategy):
    min_pullback  = 0.03
    hard_stop     = 0.08
    cooldown_bars = 7
    TSL           = 0.05
    
    def init(self, *args, **kwargs):
        self.i_pullback = self.I(ret_series, self.data.pullback_20)
        self.i_ema50    = self.I(ret_series, self.data.ema_50)
        
        self.entry_price = None
        self.current_sl  = None
        self.last_exit_bar = -100
        self.lots = 0.99
        
    def next(self):
        if not self.position:
            if len(self.data) - self.last_exit_bar < self.cooldown_bars: return
            if self.i_pullback[-1] < self.min_pullback: return
            if self.data.Close[-1] <= self.i_ema50[-1]: return
            if self.data.Close[-1] <= self.data.Open[-1]: return
            
            self.buy(size=self.lots)
            self.entry_price = self.data.Close[-1]
            self.current_sl  = self.entry_price * (1 - self.hard_stop)
            
        elif self.entry_price is not None:
            price = self.data.Close[-1]
            pnl_pct = (price - self.entry_price) / self.entry_price
            
            if pnl_pct >= self.TSL:
                new_sl = price * (1 - self.TSL)
                if new_sl > self.current_sl:
                    self.current_sl = new_sl
            
            if price < self.current_sl:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return
                
            if price < self.i_ema50[-1]:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return


# ==============================================================================
# 5. ЗАПУСК И ПРОВЕРКА
# ==============================================================================
# ==============================================================================
# 5. ЗАПУСК
# ==============================================================================
bt_new = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats_new = bt_new.run()
bt_new.plot()

trades = stats_new['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats_new.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats_new.get('Max. Drawdown [%]', 0):.1f}%")
    
    # Покажи, сколько сделок закрылось по каскадному трейлу
    # (в backtesting.py нет прямой метки, но можно оценить по длительности и профиту)
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.12)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-12%): {len(spike_trades)}")

print("✅ Бэктест прошёл без ошибок")
print(f"📊 Прибыльность: {stats_new['Return [%]']:.2f}%")
print(f"📈 Сделок: {stats_new['# Trades']}")
print("🎉 Адаптер backtesting.py работает корректно.")

2026-04-14 18:56:52,078 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-14 18:56:52,189 | INFO | TConnector | 📥 Кусок [04-15 18:56 - 04-15 18:56]: 239 свечей. Всего: 239
2026-04-14 18:56:52,518 | INFO | TConnector | 📥 Кусок [04-15 18:56 - 04-15 18:56]: 252 свечей. Всего: 491
2026-04-14 18:56:52,810 | INFO | TConnector | 📥 Кусок [04-15 18:56 - 04-14 18:56]: 253 свечей. Всего: 744
2026-04-14 18:56:53,133 | INFO | TConnector | 📥 Кусок [04-14 18:56 - 04-14 18:56]: 266 свечей. Всего: 1010
2026-04-14 18:56:53,455 | INFO | TConnector | 📥 Кусок [04-14 18:56 - 04-14 18:56]: 329 свечей. Всего: 1339
2026-04-14 18:56:53,658 | INFO | TConnector | ✅ Загрузка для SBER завершена. Всего: 1339 свечей


📊 Сделок: 13
💰 Win rate: 38.5%
🔁 Profit Factor: 2.29
📈 Total Return: 33.1%
📉 Max Drawdown: -24.2%
🎯 Сделок в диапазоне 'скачка' (3-12%): 2
✅ Бэктест прошёл без ошибок
📊 Прибыльность: 33.14%
📈 Сделок: 13
🎉 Адаптер backtesting.py работает корректно.


In [49]:
df

,Open,High,Low,Close,Volume,pullback_20,ema_50
2021-04-15,229.500,233.525,228.225,232.125,7923958,0.000000,232.125000
2021-04-16,233.775,237.700,232.825,237.300,10450590,0.000000,232.327941
2021-04-19,238.000,239.000,231.750,233.375,10780654,0.016540,232.369002
2021-04-20,233.375,241.000,229.625,231.525,27377882,0.024336,232.335904
2021-04-21,233.825,239.750,232.475,237.475,13560194,0.000000,232.537437
...,...,...,...,...,...,...,...
2026-04-10,90.150,91.250,89.550,90.400,46029849,0.022756,86.157071
2026-04-11,90.425,90.810,90.400,90.800,2498528,0.018431,86.339147
2026-04-12,90.800,91.000,90.500,90.840,4661094,0.017999,86.515651
2026-04-13,90.840,92.485,90.840,91.955,69225123,0.005946,86.728958


In [63]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from backtesting import Backtest
from strategy.strategies.trailing_trend import TrailingTrendStrategy
#from adapters.backtesting import BTAdapter
from strategy.adapters.backtesting import create_bt_adapter

df = df_raw.copy()

df = df.dropna()  # backtesting не любит NaN

# 2. Инициализируем ядро стратегии
core = TrailingTrendStrategy(
    params={"min_pullback": 0.03, "hard_stop": 0.08, "TSL": 0.05, "cooldown_bars":7},
    direction="BUY_ONLY"
)



# Рассчитываем метрики через обёртку
df['pullback_20'] = calculate_metric_series(df, Pullback20Metric, window=20, metric_key='pullback_20')
df['ema_50']      = calculate_metric_series(df, EMA50Metric, window=100, metric_key='ema_50')



bt = Backtest(df, create_bt_adapter(core), cash=100_000, commission=0.0015, trade_on_close=True)
stats = bt.run()

print("✅ Бэктест прошёл без ошибок")
print(f"📊 Прибыльность: {stats['Return [%]']:.2f}%")
print(f"📈 Сделок: {stats['# Trades']}")
print("🎉 Адаптер backtesting.py работает корректно.")

✅ Бэктест прошёл без ошибок
📊 Прибыльность: 85.99%
📈 Сделок: 34
🎉 Адаптер backtesting.py работает корректно.


In [64]:
bt.plot()

GridPlot(id='p10879', ...)

In [65]:
stats['_trades']

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag
0,171,2,11,582.6,600.1,None,None,2689.13745,303.36255,0.026993,2021-04-19,2021-04-30,11 days,None
1,161,18,57,635.0,819.0,None,None,29272.85900,351.14100,0.286329,2021-05-12,2021-07-06,55 days,None
2,163,65,89,806.0,807.7,None,None,-117.44965,394.54965,-0.000894,2021-07-16,2021-08-19,34 days,None
3,150,96,123,876.9,797.1,None,None,-12346.65000,376.65000,-0.093866,2021-08-30,2021-10-06,37 days,None
4,140,132,194,848.7,728.1,None,None,-17215.12800,331.12800,-0.144887,2021-10-19,2022-01-18,91 days,None
5,133,201,218,767.3,656.0,None,None,-15086.84835,283.94835,-0.147837,2022-01-27,2022-02-21,25 days,None
6,176,239,248,492.0,447.1,None,None,-8150.32240,247.92240,-0.094123,2022-04-21,2022-05-06,15 days,None
7,174,255,263,452.9,404.0,None,None,-8732.25090,223.65090,-0.110809,2022-05-19,2022-05-31,12 days,None
8,193,274,278,363.0,362.0,None,None,-402.88750,209.88750,-0.005751,2022-06-16,2022-06-22,6 days,None
9,176,285,292,395.0,413.8,None,None,3095.27680,213.52320,0.044524,2022-07-01,2022-07-12,11 days,None


In [101]:
# 🔍 Быстрая проверка сходимости
print("📊 Проверка сходимости метрик (последние 5 баров):")
benchmark_pullback = df_raw['close'].rolling(20, min_periods=1).max()
benchmark_pullback = (benchmark_pullback - df_raw['close']) / benchmark_pullback
check_df = pd.DataFrame({
    'my_pullback': df2['pullback_20'],
    'pd_pullback': benchmark_pullback,
    'my_ema': df2['ema_50'],
    'pd_ema': df_raw['close'].ewm(span=50, adjust=False).mean()
}).tail()
print(check_df.round(4))

📊 Проверка сходимости метрик (последние 5 баров):


ValueError: cannot reindex on an axis with duplicate labels

In [171]:
df1['Close']


2022-01-11    1822.01
2022-01-12    1825.32
2022-01-13    1822.09
2022-01-14    1817.22
2022-01-17    1818.99
               ...   
2026-04-07    4706.39
2026-04-08    4719.48
2026-04-09    4766.56
2026-04-10    4749.58
2026-04-11    4750.47
Name: Close, Length: 1217, dtype: float64

In [168]:
df2

,Open,High,Low,Close,Volume,pullback_20,ema_50
2021-08-30,1817.55,1823.38,1807.28,1810.17,0,0.0036,1797.4051
2021-08-31,1810.26,1819.43,1801.47,1813.43,0,0.0018,1797.8902
2021-09-01,1813.57,1820.21,1808.18,1813.66,0,0.0017,1799.0067
2021-09-02,1813.92,1817.53,1804.48,1809.40,0,0.0040,1799.6541
2021-09-03,1809.98,1834.55,1808.53,1826.19,0,0.0000,1800.5666
...,...,...,...,...,...,...,...
2026-04-07,4649.66,4719.86,4604.80,4706.39,0,0.0129,4778.3430
2026-04-08,4706.30,4855.11,4700.19,4719.48,0,0.0102,4776.2057
2026-04-09,4719.64,4800.72,4696.31,4766.56,0,0.0003,4775.1786
2026-04-10,4766.50,4795.34,4732.48,4749.58,0,0.0039,4774.2462


In [157]:
trades = stats['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats.get('Max. Drawdown [%]', 0):.1f}%")
    
    # Покажи, сколько сделок закрылось по каскадному трейлу
    # (в backtesting.py нет прямой метки, но можно оценить по длительности и профиту)
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.12)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-12%): {len(spike_trades)}")

📊 Сделок: 35
💰 Win rate: 37.1%
🔁 Profit Factor: 1.69
📈 Total Return: 48.6%
📉 Max Drawdown: -30.4%
🎯 Сделок в диапазоне 'скачка' (3-12%): 3


In [53]:
df.loc["2021-05-07"]

Open          615.900000
High          645.000000
Low           601.400000
Close         642.400000
Volume      36190.000000
pullback        0.000000
EMA50         593.915794
Name: 2021-05-07 00:00:00, dtype: float64

In [67]:
df['pullback']

time
2021-04-13    0.000000
2021-04-14    0.013922
2021-04-15    0.000849
2021-04-16    0.007301
2021-04-19    0.010866
                ...   
2026-04-08    0.033878
2026-04-09    0.045004
2026-04-10    0.044933
2026-04-11    0.041866
2026-04-12    0.039350
Name: pullback, Length: 1315, dtype: float64

In [68]:
df_raw

,open,high,low,close,volume
time,,,,,
2021-04-13,589.1,610.0,589.0,589.0,8400
2021-04-14,580.4,596.7,580.3,580.8,15570
2021-04-15,590.0,598.0,583.6,588.5,19290
2021-04-16,593.1,593.1,581.0,584.7,7470
2021-04-19,581.0,594.0,581.0,582.6,14190
...,...,...,...,...,...
2026-04-08,1374.0,1385.0,1341.1,1354.6,211108
2026-04-09,1354.6,1363.4,1323.0,1339.0,115233
2026-04-10,1339.0,1349.0,1320.5,1339.1,72535


In [49]:
stats_new['_trades']

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_ret_series(pullback_…),Exit_ret_series(pullback_…),Entry_ret_series(ema_50),Exit_ret_series(ema_50)
0,121,19,43,813.7,814.9,None,None,-150.39090,295.59090,-0.001527,2021-07-19,2021-08-20,32 days,None,0.1274,0.0608,739.22,792.56
1,113,64,72,867.0,831.9,None,None,-4254.26355,287.96355,-0.043424,2021-09-20,2021-09-30,10 days,None,0.0634,0.0741,844.84,843.49
2,230,250,251,410.0,400.0,None,None,-2579.45000,279.45000,-0.027354,2022-07-19,2022-07-20,1 days,None,0.0969,0.1115,404.77,408.80
3,219,261,264,418.1,442.9,None,None,5148.36150,282.83850,0.056227,2022-08-03,2022-08-08,5 days,None,0.0655,0.0557,406.58,409.67
4,206,282,296,470.0,394.0,None,None,-15922.97600,266.97600,-0.164460,2022-09-01,2022-09-21,20 days,None,0.0197,0.1508,433.71,449.59
5,183,312,331,443.0,457.1,None,None,2333.22255,247.07745,0.028781,2022-10-13,2022-11-10,28 days,None,0.0945,0.0425,428.55,451.83
6,180,342,344,462.9,448.6,None,None,-2820.10500,246.10500,-0.033846,2022-11-25,2022-11-29,4 days,None,0.0528,0.0834,454.89,447.26
7,180,402,403,448.5,451.0,None,None,207.13500,242.86500,0.002566,2023-02-20,2023-02-21,1 days,None,0.0761,0.0663,446.46,446.15
8,143,438,453,565.5,521.7,None,None,-6496.60440,233.20440,-0.080337,2023-04-13,2023-05-04,21 days,None,0.0598,0.1068,502.18,522.79
9,133,460,492,557.5,638.2,None,None,10494.55785,238.54215,0.141536,2023-05-16,2023-06-30,45 days,None,0.0209,0.0634,531.73,605.36


In [314]:
df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="SBER",
    timeframe="1d",  # Дневки: ~1000 строк за 4 года — идеально для бэктеста
    years=5
)

df = df_raw.copy()

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})

# ===== 3. СТРАТЕГИЯ: "DUMB & ROBUST" + TSL =====
def ret_series(s): return pd.Series(s)

class TrailingTrendStrategy(Strategy):
    # ПАРАМЕТРЫ
    #min_pullback  = 0.01
    #hard_stop     = 0.05
    #cooldown_bars = 10
    #TSL           = 0.05  # Trailing Stop Loss: 5% от текущей цены
    std_pct_filter = 0.05
    period_pullback = 10
    #period_sma = 50
    period_std = 20
    period = 50

        # ===== 1. МИНИМУМ МЕТРИК =====
    def calculate_pullback(series, window=20):
        rolling_max = series.rolling(window=window, min_periods=1).max()
        return (rolling_max - series) / rolling_max
    
    # ===== 2. РАСЧЁТ =====
    df['pullback'] = calculate_pullback(df['Close'], period_pullback)
    typical = (df['High']+df['Low'])/2
    df['SMA50']    = typical.rolling(window=period, min_periods=1).mean()
    df['std_pct']    = typical.rolling(window=period_std, min_periods=1).std()/df['Close']
    
    
    #df = df.loc[df.index>"2022-01-10"]
    df = df.dropna(subset=['SMA50','pullback'])
    
    def init(self, *args, **kwargs):
        self.i_sma50    = self.I(ret_series, self.data.SMA50)
        self.i_std_pct = self.I(ret_series, self.data.std_pct)
        self.i_pullback = self.I(ret_series, self.data.pullback)
        self.entry_price = None
        self.current_sl  = None  # Текущий уровень стопа (динамический)
        self.last_exit_bar = -100
        self.lots = 0.99
        

    def next(self):

        self.hard_stop = self.i_std_pct[-1] if self.i_std_pct[-1] > self.std_pct_filter else self.std_pct_filter
        self.TSL = self.hard_stop
        self.cooldown_bars = self.period_pullback
        self.min_pullback = 0.01
        
        
        # === ВХОД ===
        if not self.position:
            # 1. Кулдаун
            if len(self.data) - self.last_exit_bar < self.cooldown_bars:
                return
            # 2. Откат
            if self.i_pullback[-1] < self.min_pullback: return
            # 3. Цена выше SMA50
            if self.data.Close[-1] <= self.i_sma50[-1]: return
            # 4. Зелёная свеча
            if self.data.Close[-1] <= self.data.Open[-1]: return
            # Фильтр флэта
            if self.i_std_pct[-1] < self.std_pct_filter: return
                
            
            self.buy(size=self.lots)
            self.entry_price = self.data.Close[-1]
            self.current_sl  = self.entry_price * (1 - self.hard_stop)  # Инициализируем жёсткий стоп
            
        # === ВЫХОД / УПРАВЛЕНИЕ ПОЗИЦИЕЙ ===
        elif self.entry_price is not None:
            price = self.data.Close[-1]
            pnl_pct = (price - self.entry_price) / self.entry_price
            
            # 🔥 ЛОГИКА TRAILING STOP
            # Как только профит >= TSL (5%), активируем трейл
            if pnl_pct >= self.TSL:
                # Рассчитываем новый стоп: текущая цена - TSL%
                new_sl = price * (1 - self.TSL)
                # Двигаем стоп ТОЛЬКО вверх
                if new_sl > self.current_sl:
                    self.current_sl = new_sl
            
            # 1. Выход по трейлинг-стопу (или начальному hard stop)
            if price < self.current_sl:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return
                
            # 2. Выход по пробою EMA50 вниз (резервный фильтр)
            if price < self.i_sma50[-1]:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.last_exit_bar = len(self.data)
                return

# ===== 4. ЗАПУСК =====
bt = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

trades = stats['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats.get('Max. Drawdown [%]', 0):.1f}%")
    
    # Покажи, сколько сделок закрылось по каскадному трейлу
    # (в backtesting.py нет прямой метки, но можно оценить по длительности и профиту)
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.12)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-12%): {len(spike_trades)}")

2026-04-13 09:10:15,440 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-13 09:10:15,524 | INFO | TConnector | 📥 Кусок [04-14 09:10 - 04-14 09:10]: 239 свечей. Всего: 239
2026-04-13 09:10:15,827 | INFO | TConnector | 📥 Кусок [04-14 09:10 - 04-14 09:10]: 253 свечей. Всего: 492
2026-04-13 09:10:16,155 | INFO | TConnector | 📥 Кусок [04-14 09:10 - 04-13 09:10]: 254 свечей. Всего: 746
2026-04-13 09:10:16,437 | INFO | TConnector | 📥 Кусок [04-13 09:10 - 04-13 09:10]: 265 свечей. Всего: 1011
2026-04-13 09:10:16,732 | INFO | TConnector | 📥 Кусок [04-13 09:10 - 04-13 09:10]: 329 свечей. Всего: 1340
2026-04-13 09:10:16,934 | INFO | TConnector | ✅ Загрузка для SBER завершена. Всего: 1340 свечей


📊 Сделок: 5
💰 Win rate: 60.0%
🔁 Profit Factor: 2.39
📈 Total Return: 12.1%
📉 Max Drawdown: -8.8%
🎯 Сделок в диапазоне 'скачка' (3-12%): 2


In [316]:
# ===== 5. ОПТИМИЗАЦИЯ =====
bt = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)

stats, heatmap = bt.optimize(
    #min_pullback=np.arange(0.01, 0.05, 0.01).tolist(),
    #hard_stop=np.arange(0.050, 0.100, 0.01).tolist(),
    #cooldown_bars=np.arange(3, 10, 1).tolist(),
    #std_pct_filter=np.arange(0.015, 0.050, 0.005).tolist(),
    period=np.arange(10, 100, 10).tolist(),
    period_pullback = np.arange(10, 100, 10).tolist(),
    #period_sma = 50
    period_std = np.arange(10, 100, 10).tolist(),



    # Логическое ограничение: стоп должен быть шире зоны отката, иначе будет постоянный вынос
    #constraint=lambda p: p.hard_stop > p.min_pullback + 0.015,
    
    # Максимизируем соотношение Доходность / Просадка (именно то, что тебе нужно)
    maximize=lambda s: s['Return [%]'] / max(abs(s['Max. Drawdown [%]']), 1),
    
    return_heatmap=True
)

print("\n🏆 Метрики лучшей комбинации:")
print(stats)

# После optimize() backtest автоматически переключается на лучшие параметры
bt.plot()

C:\Users\Khanin Maksim\AppData\Local\Programs\Python\Python312\Lib\site-packages\backtesting\backtesting.py:1624: UserWarning: Searching for best of 729 configurations.
  output = _optimize_grid()
C:\Users\Khanin Maksim\AppData\Local\Programs\Python\Python312\Lib\site-packages\backtesting\backtesting.py:1624: RuntimeWarning: If you want to use multi-process optimization with `multiprocessing.get_start_method() == 'spawn'` (e.g. on Windows),set `backtesting.Pool = multiprocessing.Pool` (or of the desired context) and hide `bt.optimize()` call behind a `if __name__ == '__main__'` guard. Currently using thread-based paralellism, which might be slightly slower for non-numpy / non-GIL-releasing code. See https://github.com/kernc/backtesting.py/issues/1256
  output = _optimize_grid()



🏆 Метрики лучшей комбинации:
Start                     2021-04-14 00:00:00
End                       2026-04-13 00:00:00
Duration                   1825 days 00:00:00
Exposure Time [%]                    10.74627
Equity Final [$]                 112126.35125
Equity Peak [$]                  118870.44495
Commissions [$]                    1444.72875
Return [%]                           12.12635
Buy & Hold Return [%]                12.28237
Return (Ann.) [%]                     2.18073
Volatility (Ann.) [%]                  8.8718
CAGR [%]                              1.59299
Sharpe Ratio                           0.2458
Sortino Ratio                         0.40961
Calmar Ratio                          0.24686
Alpha [%]                            11.43614
Beta                                  0.05619
Max. Drawdown [%]                    -8.83375
Avg. Drawdown [%]                    -3.83499
Max. Drawdown Duration      651 days 00:00:00
Avg. Drawdown Duration      163 days 00:00:00
# Tr

GridPlot(id='p73376', ...)

In [320]:
df['Close']

2021-04-14    288.30
2021-04-15    283.17
2021-04-16    289.03
2021-04-19    287.84
2021-04-20    287.92
               ...  
2026-04-09    317.51
2026-04-10    317.29
2026-04-11    317.91
2026-04-12    317.64
2026-04-13    317.95
Name: Close, Length: 1340, dtype: float64

In [288]:
!pip install openpyxl
pd.DataFrame(heatmap).to_csv("output2.csv")

In [231]:
df_raw = await fetch_candles_to_df(
    connector=connector,
    ticker="MDMG",
    timeframe="1d",
    years=5
)

df = df_raw.copy()

# ===== 1. МИНИМУМ МЕТРИК =====
def calculate_pullback(series, window=20):
    rolling_max = series.rolling(window=window, min_periods=1).max()
    return (rolling_max - series) / rolling_max

# ===== 2. РАСЧЁТ =====
df['pullback'] = calculate_pullback(df['close'], 20)
typical = (df['high']+df['low'])/2
df['SMA50']    = typical.rolling(window=50, min_periods=1).mean()
# ИСПРАВЛЕНО: корректное поэлементное деление, а не на скаляр df['close'][-1]
df['std_pct']  = df['close'].rolling(window=50, min_periods=1).std() / df['close']

df = df.rename(columns={'open':'Open','high':'High','low':'Low','close':'Close','volume':'Volume'})
df = df.dropna(subset=['SMA50','pullback'])

# ===== 3. СТРАТЕГИЯ: "DUMB & ROBUST" + STEP TSL + TIME STOP =====
def ret_series(s): return pd.Series(s)

class TrailingTrendStrategy(Strategy):
    # ПАРАМЕТРЫ (сбалансированы под DD <15% и PF >1.7)
    min_pullback  = 0.035
    hard_stop     = 0.06      # Жёсткий стоп 6% (реже выбивает шумом, чем 8%)
    cooldown_bars = 5
    lots = 0.99
    std_pct_filter = 0.02
    time_stop_days = 15       # Макс. удержание позиции (убирает застойные DD)
    
    def init(self, *args, **kwargs):
        self.i_sma50    = self.I(ret_series, self.data.SMA50)
        self.i_std_pct  = self.I(ret_series, self.data.std_pct)
        self.i_pullback = self.I(ret_series, self.data.pullback)
        
        self.entry_price = None
        self.current_sl  = None
        self.peak_price  = None
        self.entry_bar   = None
        self.last_exit_bar = -100

    def next(self):
        # === ВЫХОД / УПРАВЛЕНИЕ ПОЗИЦИЕЙ ===
        if self.position and self.entry_price is not None:
            price = self.data.Close[-1]
            pnl_pct = (price - self.entry_price) / self.entry_price
            
            # Фиксируем локальный максимум
            if price > self.peak_price:
                self.peak_price = price
                
            # 🔥 ПОЭТАПНЫЙ ТРЕЙЛИНГ (STEP TRAIL)
            if pnl_pct >= 0.02:
                # 2%+ профита → переносим стоп в безубыток
                self.current_sl = max(self.current_sl, self.entry_price)
            if pnl_pct >= 0.05:
                # 5%+ профита → включаем плотный трейл 4% от пика
                new_sl = self.peak_price * (1 - 0.04)
                if new_sl > self.current_sl:
                    self.current_sl = new_sl

            # 1. Выход по стопу
            if price < self.current_sl:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.peak_price  = None
                self.last_exit_bar = len(self.data)
                return
                
            # 2. Выход по времени (Time Stop)
            if len(self.data) - self.entry_bar > self.time_stop_days:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.peak_price  = None
                self.last_exit_bar = len(self.data)
                return

            # 3. Выход по SMA50 ТОЛЬКО если сделка в минусе
            if pnl_pct < 0 and price < self.i_sma50[-1]:
                self.position.close()
                self.entry_price = None
                self.current_sl  = None
                self.peak_price  = None
                self.last_exit_bar = len(self.data)
                return

        # === ВХОД ===
        if not self.position:
            if len(self.data) - self.last_exit_bar < self.cooldown_bars:
                return
            if self.i_pullback[-1] < self.min_pullback: return
            if self.data.Close[-1] <= self.i_sma50[-1]: return
            if self.data.Close[-1] <= self.data.Open[-1]: return
            if self.i_std_pct[-1] < self.std_pct_filter: return
                
            self.buy(size=self.lots)
            self.entry_price = self.data.Close[-1]
            self.peak_price  = self.entry_price
            self.entry_bar   = len(self.data)
            self.current_sl  = self.entry_price * (1 - self.hard_stop)

# ===== 4. ЗАПУСК =====
bt = Backtest(df, TrailingTrendStrategy, cash=100_000, commission=0.0015, finalize_trades=True)
stats = bt.run()
bt.plot()

trades = stats['_trades']
if trades.empty:
    print("❌ Нет сделок")
else:
    ret_col = 'ReturnPct' if 'ReturnPct' in trades.columns else 'Return%'
    print(f"📊 Сделок: {len(trades)}")
    print(f"💰 Win rate: {(trades[ret_col] > 0).mean():.1%}")
    wins = trades[trades[ret_col] > 0][ret_col].sum()
    losses = abs(trades[trades[ret_col] < 0][ret_col].sum())
    pf = wins / losses if losses > 0 else np.inf
    print(f"🔁 Profit Factor: {pf:.2f}")
    print(f"📈 Total Return: {stats.get('Return [%]', 0):.1f}%")
    print(f"📉 Max Drawdown: {stats.get('Max. Drawdown [%]', 0):.1f}%")
    
    spike_trades = trades[(trades[ret_col] > 0.03) & (trades[ret_col] < 0.15)]
    print(f"🎯 Сделок в диапазоне 'скачка' (3-15%): {len(spike_trades)}")

2026-04-13 07:18:14,279 | INFO | TConnector | 🆕 Полная загрузка: глубина 1825 дн. (параметр)
2026-04-13 07:18:14,369 | INFO | TConnector | 📥 Кусок [04-14 07:18 - 04-14 07:18]: 235 свечей. Всего: 235
2026-04-13 07:18:14,706 | INFO | TConnector | 📥 Кусок [04-14 07:18 - 04-14 07:18]: 253 свечей. Всего: 488
2026-04-13 07:18:15,015 | INFO | TConnector | 📥 Кусок [04-14 07:18 - 04-13 07:18]: 254 свечей. Всего: 742
2026-04-13 07:18:15,318 | INFO | TConnector | 📥 Кусок [04-13 07:18 - 04-13 07:18]: 245 свечей. Всего: 987
2026-04-13 07:18:15,624 | INFO | TConnector | 📥 Кусок [04-13 07:18 - 04-13 07:18]: 327 свечей. Всего: 1314
2026-04-13 07:18:15,827 | INFO | TConnector | ✅ Загрузка для MDMG завершена. Всего: 1314 свечей


📊 Сделок: 22
💰 Win rate: 50.0%
🔁 Profit Factor: 1.05
📈 Total Return: -0.3%
📉 Max Drawdown: -19.1%
🎯 Сделок в диапазоне 'скачка' (3-15%): 6


In [227]:
stats['_trades'].to_csv()

',Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,Entry_ret_series(SMA50),Exit_ret_series(SMA50),Entry_ret_series(std_pct),Exit_ret_series(std_pct),Entry_ret_series(pullback),Exit_ret_series(pullback),Entry_ret_series(ATR),Exit_ret_series(ATR)\r\n0,11,67,91,813.7,814.9,,,-13.671900000000754,26.871900000000004,-0.001527467125476406,2021-07-19,2021-08-20,32 days,,721.9989999999999,793.805,0.09773216945050803,0.0643957749248399,0.1273957158962796,0.0608187134502924,64.50714285714285,26.814285714285713\r\n1,26,112,120,867.0,831.9,,,-978.8571000000006,66.25710000000001,-0.0434237024221454,2021-09-20,2021-09-30,10 days,,833.308,843.053,0.044074910627325777,0.040434654792221913,0.06342705335399597,0.07409489543512478,29.899999999999977,29.014285714285702\r\n2,25,298,309,410.0,418.1,,,171.44625000000056,31.05375,0.016726463414634164,2022-07-19,2022-08-03,15 days,,405.783,396.29900000000004,0.10310617175060086,0.07661853235746505,0.09686

In [224]:
df['Close'].to_csv()

',Close\r\n2021-04-14,580.8\r\n2021-04-15,588.5\r\n2021-04-16,584.7\r\n2021-04-19,582.6\r\n2021-04-20,583.3\r\n2021-04-21,583.3\r\n2021-04-22,577.6\r\n2021-04-23,571.3\r\n2021-04-26,573.3\r\n2021-04-27,636.0\r\n2021-04-28,620.2\r\n2021-04-29,615.3\r\n2021-04-30,600.1\r\n2021-05-04,590.0\r\n2021-05-05,595.1\r\n2021-05-06,613.8\r\n2021-05-07,642.4\r\n2021-05-10,628.3\r\n2021-05-11,632.2\r\n2021-05-12,635.0\r\n2021-05-13,639.6\r\n2021-05-14,635.2\r\n2021-05-17,621.4\r\n2021-05-18,632.9\r\n2021-05-19,621.2\r\n2021-05-20,620.5\r\n2021-05-21,624.3\r\n2021-05-24,629.9\r\n2021-05-25,624.9\r\n2021-05-26,627.8\r\n2021-05-27,644.3\r\n2021-05-28,647.9\r\n2021-05-31,679.5\r\n2021-06-01,693.8\r\n2021-06-02,703.9\r\n2021-06-03,713.1\r\n2021-06-04,701.0\r\n2021-06-07,709.7\r\n2021-06-08,709.0\r\n2021-06-09,703.0\r\n2021-06-10,706.2\r\n2021-06-11,718.8\r\n2021-06-14,703.8\r\n2021-06-15,700.0\r\n2021-06-16,696.7\r\n2021-06-17,700.5\r\n2021-06-18,702.0\r\n2021-06-21,725.0\r\n2021-06-22,722.0\r\n2021-06-2